# 13주차 — Navigation: 미션을 시키는 법 + 실시간 모드

```
[Sensors] → [Estimation] → [Navigation] → [Control] → [Allocation] → [Plant]
                             ▲ 이번 주
```

목표: 웨이포인트 순회(미션 계층) + 시뮬 시간 vs 벽시계 시간.

## 이론 요약
- 항법 계층은 위치 setpoint 를 만든다 — 유도(오차→속도명령)는 아래의
  위치 캐스케이드가 이미 하고 있다
- WaypointManager: 반경 안에 들어오면 다음 지점 (waypoints="N,E,D; ...")
- 실시간 모드: 배치 시뮬(수십 ms)과 수치는 같지만 벽시계에 페이싱 —
  실기체/ROS2 로 가는 전제 (P42)
- ROS2 데모 (ex6): 같은 다이어그램 구조가 실제 미들웨어의 노드가 된다

In [ ]:
# 뷰어 열기 — 뼈대 모델을 브라우저에서 편집한다.
# 완성하면 뷰어의 '저장' → week13_model.json 이 다운로드된다.
# 그 파일을 이 폴더의 같은 이름 파일에 **덮어쓰면 제출 완료** (채점 셀로 확인).
import sys, webbrowser; sys.path.insert(0, '../..')
from loopsym import load_json, render_html
p = render_html(load_json('week13_model.json'), 'week13_viewer.html',
                title='13주차', t_end=8.0, dt=0.004,
                save_name='week13_model.json')
webbrowser.open('file://' + p)

## 실습 — 미션 배선
뼈대는 hold 상수가 제자리 호버를 시키고 있다. ★wp_manager 를 살려라:
① 플랜트 위치(pN/pE/pD) → wp 입력, ② wp 의 n_sp/e_sp/d_sp → pos_ctrl.
완성하면: [2,0,−6] → [2,2,−6] → [0,0,−5] 순회 후 마지막 지점 호버.
scope_wp 의 wp_idx 가 계단처럼 0→1→2→3 오르는 것을 확인.

In [ ]:
# 노트북에서 바로 실행해 보기 (뷰어 없이) — 저장한 모델을 로드해 시뮬
import sys; sys.path.insert(0, '../..')
from loopsym import load_json, Simulator
bd = load_json('week13_model.json')
sim = Simulator(bd); sim.run(16.0, 0.004)
import matplotlib.pyplot as plt
for b in sim.bd.blocks:
    if type(b).__name__ == 'Scope':
        plt.figure(figsize=(8,2.6)); plt.title(b.name); plt.grid(True)
        for i, lab in enumerate(b.labels): plt.plot(b.t, b.data[i], label=lab)
        plt.legend()

## 실시간 모드 / ROS2 데모 (수업 시연)
```bash
# 터미널 1 — 로컬 서버 모드
python -m loopsym.server
# 뷰어 새로고침 → 배지가 'Local Server' → '실시간' 체크 → 실행
# Scope 가 벽시계 속도로 차오른다 (external mode)

# ROS2 turtlesim (환경 있는 경우): examples/ex6_ros2_turtlesim.py 참조
```

## 과제 (팀)
1. week13_model.json 완성 (채점기: 전 웨이포인트 방문 + 복귀)
2. 자기 미션 설계: 웨이포인트 4개 이상 + radius 튜닝 — 반경이 너무 작으면/크면
   무슨 일이 생기는지 리포트.

In [ ]:
# ✅ 채점 — 전부 초록색이 되면 이번 주 완료!
!python -m pytest test_week13.py -q